# Python. Лекция 14. Тестирование

Лектор: Хайбулин Даниэль

Весь семестр вы отправляли решения в CI и читали его отчёты. Эта лекция — про
то, что там происходит: как устроены тесты, как понимать их вывод и как писать
такие же самому.

Тест отвечает на один вопрос: делает ли код то, что от него ждут. Вопрос звучит
просто, но проверять руками — запустить, посмотреть глазами — можно ровно один
раз. Тест превращает разовую проверку в постоянную.

Все примеры ниже создают файлы сами и запускают на них настоящий `pytest`,
поэтому ноутбук можно запускать целиком и сколько угодно раз.

## Тест — это `assert`

Механизм примитивен. `assert` проверяет условие и, если оно ложно, поднимает
`AssertionError`:

In [ ]:
def average(values: list[float]) -> float:
    return sum(values) / len(values)


assert average([2, 4, 6]) == 4
print("сошлось")

In [ ]:
try:
    assert average([1, 2]) == 2
except AssertionError:
    print("не сошлось")

Всё тестирование стоит на этом. `pytest` не добавляет нового способа проверять —
он берёт на себя рутину: находит тесты, запускает каждый отдельно, продолжает
после падения и печатает отчёт, по которому видно, что именно разошлось.

Дальше в лекции мы будем писать файлы с тестами и запускать их по-настоящему.
Вспомогательная функция ниже делает ровно это.

In [ ]:
import subprocess
import sys
from pathlib import Path

DEMO = Path("demo")
DEMO.mkdir(exist_ok=True)


def run_pytest(*args: str) -> None:
    """Запустить pytest в каталоге demo и показать отчёт."""
    quiet = [] if "-v" in args else ["-q"]
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *quiet, "--no-header", "-p", "no:cacheprovider", *args],
        cwd=DEMO,
        capture_output=True,
        text=True,
    )
    print(result.stdout.strip())

## Как `pytest` находит тесты

Запускать тесты по одному никто не собирается: `pytest` сам обходит каталог и
собирает всё, что похоже на тест. Похоже — по имени.

In [ ]:
(DEMO / "stats.py").write_text(
    '''def average(values: list[float]) -> float:
    """Среднее арифметическое. Пустой список — ValueError."""
    if not values:
        raise ValueError("нужно хотя бы одно значение")
    return sum(values) / len(values)
''',
    encoding="utf-8",
)

(DEMO / "test_stats.py").write_text(
    """from stats import average


def test_average_of_three() -> None:
    assert average([2, 4, 6]) == 4


def test_average_of_one() -> None:
    assert average([5]) == 5
""",
    encoding="utf-8",
)

run_pytest()

Две точки — два пройденных теста. Правила поиска:

| Что | Как называется |
|---|---|
| файл | `test_*.py` или `*_test.py` |
| функция | начинается с `test_` |
| класс | начинается с `Test`, методы внутри — с `test_` |

Имя теста — не формальность: именно оно попадёт в отчёт, когда тест упадёт.
`test_average_of_one` сразу говорит, что проверялось; `test_1` не говорит
ничего.

## Как читать отчёт

Пока всё сходится, отчёт неинтересен. Сломаем реализацию — уберём деление:

In [ ]:
(DEMO / "stats.py").write_text(
    '''def average(values: list[float]) -> float:
    """Среднее арифметическое. Пустой список — ValueError."""
    if not values:
        raise ValueError("нужно хотя бы одно значение")
    return sum(values)
''',
    encoding="utf-8",
)

run_pytest()

Разбор отчёта по строкам:

* `F.` — первый тест упал, второй прошёл. `F` на каждый упавший, точка на
  пройденный;
* строка со знаком `>` — то самое место, где сработал `assert`;
* строки с `E` — что именно разошлось. `assert 12 == 4` и ниже
  `where 12 = average([2, 4, 6])`: `pytest` подставил настоящие значения вместо
  выражений. Это его главное удобство — не нужно ставить `print`, чтобы узнать,
  что пришло;
* `test_stats.py:5` — файл и строка. Отсюда начинают чинить;
* последняя строка — сводка.

Обратите внимание, что второй тест остался зелёным: `average([5])` считает
`sum([5])`, то есть 5, и без деления ответ случайно совпал. Сломанный код может
проходить часть тестов — поэтому одного случая никогда не хватает.

> **Как читать чужой трейсбек.** В отчёте бывает много строк, и почти все они
> из библиотек. Ищите первую строку, где упомянут **ваш** файл: ошибка почти
> всегда там, а не в глубине `pytest`.

### `FAILED` и `ERROR` — разные вещи

`FAILED` означает, что тест выполнялся и проверка не сошлась. `ERROR` — что до
проверки вообще не дошли: файл не импортировался, фикстура не нашлась, код упал
до `assert`.

In [ ]:
(DEMO / "test_broken.py").write_text(
    """from stats import median  # такой функции нет


def test_median() -> None:
    assert median([1, 2, 3]) == 2
""",
    encoding="utf-8",
)

run_pytest("test_broken.py")

Разница важна на практике: `FAILED` — повод смотреть на логику, `ERROR` — почти
всегда опечатка в импорте, забытый файл или неверное имя фикстуры. Чинить их
надо первыми: пока файл не импортируется, ни один тест в нём не проверен.

In [ ]:
(DEMO / "test_broken.py").unlink()

## Что проверять

Самый частый вопрос — сколько тестов писать. Ориентир простой: по одному на
каждый разный **вид** входа.

| Вид | Для `average` |
|---|---|
| обычный случай | несколько чисел |
| граница | ровно одно число |
| пустой вход | пустой список |
| ошибочный вход | не число в списке |

Тест, повторяющий уже проверенное с другими числами, не добавляет ничего:
`average([2, 4, 6])` и `average([10, 20, 30])` проверяют одно и то же.

## Параметризация

Когда случаев много, писать функцию на каждый — лишняя работа. Декоратор
`parametrize` разворачивает один тест в несколько:

In [ ]:
(DEMO / "stats.py").write_text(
    '''def average(values: list[float]) -> float:
    """Среднее арифметическое. Пустой список — ValueError."""
    if not values:
        raise ValueError("нужно хотя бы одно значение")
    return sum(values) / len(values)
''',
    encoding="utf-8",
)

(DEMO / "test_stats.py").write_text(
    """import pytest
from stats import average

cases = [
    ([2, 4, 6], 4),
    ([5], 5),
    ([-1, 1], 0),
    ([1, 2], 1.5),
]


@pytest.mark.parametrize("values, expected", cases)
def test_average(values: list[float], expected: float) -> None:
    assert average(values) == expected
""",
    encoding="utf-8",
)

run_pytest("-v")

Каждый случай стал отдельным тестом со своим именем. В скобках `pytest`
показывает параметры: простые значения он пишет как есть, а списки нумерует —
`values0`, `values1`. Когда имена хочется читаемые, их задают явно:
`@pytest.mark.parametrize(..., ids=["три числа", "одно", "симметрично", "два"])`.

Отсюда же следует, почему цикл внутри теста хуже: он падает на первом
несошедшемся случае, и остальные остаются непроверенными.

In [ ]:
(DEMO / "test_loop.py").write_text(
    """from stats import average

cases = [([2, 4, 6], 999), ([5], 888), ([-1, 1], 777)]


def test_average_in_a_loop() -> None:
    for values, expected in cases:
        assert average(values) == expected
""",
    encoding="utf-8",
)

run_pytest("test_loop.py")

Один упавший тест вместо трёх, и про два оставшихся случая мы ничего не узнали.
С `parametrize` отчёт показал бы все три.

In [ ]:
(DEMO / "test_loop.py").unlink()

## Ожидаемые исключения

Иногда правильное поведение — это падение. Проверяют его менеджером контекста
`pytest.raises`: он ждёт исключение и считает тест пройденным, если оно
случилось.

In [ ]:
(DEMO / "test_errors.py").write_text(
    """import pytest
from stats import average


def test_average_of_nothing() -> None:
    with pytest.raises(ValueError):
        average([])
""",
    encoding="utf-8",
)

run_pytest("test_errors.py")

Две тонкости.

Внутрь `with` кладут **только** ту строку, которая должна упасть. Если завернуть
туда половину теста, исключение может прийти совсем из другого места, а тест всё
равно позеленеет.

Второе: `pytest.raises(Exception)` пройдёт почти на любой поломке, включая
опечатку в имени переменной. Указывают конкретный класс, а когда важен текст —
добавляют `match`:

In [ ]:
(DEMO / "test_errors.py").write_text(
    """import pytest
from stats import average


def test_average_of_nothing() -> None:
    with pytest.raises(ValueError, match="хотя бы одно"):
        average([])
""",
    encoding="utf-8",
)

run_pytest("test_errors.py")

## Сравнение чисел с плавающей точкой

Отдельная ловушка, о которой спотыкаются все. Дробные числа хранятся неточно, и
привычное равенство не работает:

In [ ]:
print(0.1 + 0.2 == 0.3)
print(f"{0.1 + 0.2:.20f}")

Сравнивать такие числа нужно с допуском. В `pytest` для этого есть `approx`:

In [ ]:
(DEMO / "test_float.py").write_text(
    """import pytest


def test_naive() -> None:
    assert 0.1 + 0.2 == 0.3


def test_with_approx() -> None:
    assert 0.1 + 0.2 == pytest.approx(0.3)
""",
    encoding="utf-8",
)

run_pytest("test_float.py")

`approx` работает и со списками, и со словарями: `[0.1 + 0.2, 0.5] ==
pytest.approx([0.3, 0.5])`.

> **Типичная ошибка.** Тест на дробных числах, который проходит на одной машине
> и падает на другой. Причина почти всегда эта: сравнение точным равенством там,
> где нужен допуск.

In [ ]:
(DEMO / "test_float.py").unlink()

## Фикстуры

Тесту часто нужна подготовленная обстановка: файл, объект, каталог. Фикстура —
это способ такую обстановку описать один раз и попросить по имени.

### `tmp_path`

Самая полезная фикстура для того, чем мы занимались в прошлой лекции. Код,
который читает файлы, нельзя проверять на настоящих файлах проекта: тест будет
зависеть от того, что лежит на диске, и сможет их испортить.

`tmp_path` даёт тесту пустой временный каталог — свой на каждый тест. Достаточно
назвать его в аргументах:

In [ ]:
(DEMO / "scores.py").write_text(
    '''import csv
from pathlib import Path


def load(path: Path) -> dict[str, int]:
    """Прочитать csv «имя,балл» в словарь."""
    with path.open(encoding="utf-8", newline="") as file:
        return {row["имя"]: int(row["балл"]) for row in csv.DictReader(file)}
''',
    encoding="utf-8",
)

(DEMO / "test_scores.py").write_text(
    """from pathlib import Path

from scores import load


def test_load(tmp_path: Path) -> None:
    path = tmp_path / "баллы.csv"
    path.write_text("имя,балл\\nИванов,7\\nПетрова,9\\n", encoding="utf-8")
    assert load(path) == {"Иванов": 7, "Петрова": 9}


def test_load_converts_to_int(tmp_path: Path) -> None:
    path = tmp_path / "баллы.csv"
    path.write_text("имя,балл\\nИванов,7\\n", encoding="utf-8")
    assert load(path)["Иванов"] + 1 == 8
""",
    encoding="utf-8",
)

run_pytest("test_scores.py")

Каталог создаётся до теста и убирается после. Тесты не мешают друг другу, потому
что каталоги у них разные.

### Своя фикстура

Фикстуру объявляют декоратором `@pytest.fixture`. Имя функции становится именем
аргумента:

In [ ]:
(DEMO / "test_fixture.py").write_text(
    """from pathlib import Path

import pytest
from scores import load


@pytest.fixture
def table(tmp_path: Path) -> Path:
    path = tmp_path / "баллы.csv"
    path.write_text("имя,балл\\nИванов,7\\nПетрова,9\\n", encoding="utf-8")
    return path


def test_names(table: Path) -> None:
    assert sorted(load(table)) == ["Иванов", "Петрова"]


def test_total(table: Path) -> None:
    assert sum(load(table).values()) == 16
""",
    encoding="utf-8",
)

run_pytest("test_fixture.py")

Фикстуры складываются: `table` сама попросила `tmp_path`. Так строят обстановку
по частям, не повторяя подготовку в каждом тесте.

### Уборка и область видимости

Если фикстура не возвращает значение, а отдаёт его через `yield`, то код после
`yield` выполнится **после** теста. Туда кладут уборку: закрыть файл, оборвать
соединение.

По умолчанию фикстура создаётся заново для каждого теста. Параметр `scope`
меняет это: `scope="module"` — одна на файл, `scope="session"` — одна на весь
прогон. Общая фикстура быстрее, но если тест её испортит, следующий получит
испорченную.

In [ ]:
(DEMO / "test_scope.py").write_text(
    """from collections.abc import Iterator

import pytest


@pytest.fixture(scope="module")
def connection() -> Iterator[str]:
    print("\\nоткрыли соединение")
    yield "соединение"
    print("\\nзакрыли соединение")


def test_first(connection: str) -> None:
    assert connection == "соединение"


def test_second(connection: str) -> None:
    assert connection == "соединение"
""",
    encoding="utf-8",
)

run_pytest("test_scope.py", "-s")

Открыли один раз на два теста, закрыли в конце.

### `conftest.py`

Фикстуры, нужные нескольким файлам, выносят в `conftest.py` рядом с тестами:
`pytest` подхватывает его сам, импортировать ничего не нужно. Именно этот файл
лежит в каждой задаче курса — в нём одна строка, добавляющая каталог задачи в
пути поиска модулей.

## `monkeypatch`

Иногда проверяемый код зависит от внешнего: переменной окружения, текущего
времени, текущего каталога. Тест не должен от этого зависеть — но и менять
настройки машины он не имеет права.

Фикстура `monkeypatch` подменяет что-нибудь **на время одного теста** и сама
возвращает как было:

In [ ]:
(DEMO / "settings.py").write_text(
    '''import os
from pathlib import Path


def reports_dir() -> Path:
    """Каталог отчётов: из переменной окружения, иначе рядом с программой."""
    return Path(os.environ.get("REPORTS", "reports"))
''',
    encoding="utf-8",
)

(DEMO / "test_settings.py").write_text(
    """from pathlib import Path

import pytest
from settings import reports_dir


def test_default() -> None:
    assert reports_dir() == Path("reports")


def test_from_environment(monkeypatch: pytest.MonkeyPatch) -> None:
    monkeypatch.setenv("REPORTS", "/data/отчёты")
    assert reports_dir() == Path("/data/отчёты")


def test_default_again() -> None:
    # Значение из предыдущего теста откатилось само.
    assert reports_dir() == Path("reports")
""",
    encoding="utf-8",
)

run_pytest("test_settings.py")

Третий тест — главное в этом примере: подмена жила ровно один тест. Кроме
`setenv` у `monkeypatch` есть `delenv`, `setattr` для подмены функции или
атрибута и `chdir` для смены текущего каталога — того самого, от которого, как
мы разбирали в прошлой лекции, считаются относительные пути.

## Плохие тесты

Зелёный прогон сам по себе ничего не гарантирует: тест может не проверять
ничего. Три случая, которые встречаются чаще всего.

### Тест без проверки

Самый опасный: он проходит всегда, потому что в нём нет `assert`.

In [ ]:
(DEMO / "stats.py").write_text(
    '''def average(values: list[float]) -> float:
    """Заведомо сломано: ничего не возвращает."""
    if not values:
        raise ValueError("нужно хотя бы одно значение")
''',
    encoding="utf-8",
)

(DEMO / "test_useless.py").write_text(
    """from stats import average


def test_average_works() -> None:
    average([2, 4, 6])
""",
    encoding="utf-8",
)

run_pytest("test_useless.py")

Функция не возвращает ничего, а тест зелёный: он вызвал её и не спросил
результат. Проверка — не «код выполнился», а «результат такой, какой нужен».

Тот же изъян бывает тоньше: тест проверяет то, что и так дано. Утверждение
`assert isinstance(average, type(average))` или проверка константы, объявленной
в шаблоне, пройдут на нерешённой задаче.

> **Как поймать.** Сломайте проверяемый код нарочно и запустите тест. Если он
> остался зелёным — он ничего не проверял.

In [ ]:
(DEMO / "test_useless.py").unlink()

### Тест на реализацию, а не на поведение

Такой тест ломается от безобидной правки. Классический пример — проверка
точного текста сообщения об ошибке:

In [ ]:
(DEMO / "stats.py").write_text(
    '''def average(values: list[float]) -> float:
    """Среднее арифметическое. Пустой список — ValueError."""
    if not values:
        raise ValueError("список значений пуст")
    return sum(values) / len(values)
''',
    encoding="utf-8",
)

(DEMO / "test_wording.py").write_text(
    """import pytest
from stats import average


def test_message_text() -> None:
    with pytest.raises(ValueError) as info:
        average([])
    assert str(info.value) == "нужно хотя бы одно значение"
""",
    encoding="utf-8",
)

run_pytest("test_wording.py")

Поведение не изменилось — функция по-прежнему отказывается считать среднее от
пустого списка, — а тест красный: переформулировали сообщение. Проверять надо
класс исключения, а от текста брать в лучшем случае устойчивый кусок через
`match`.

In [ ]:
(DEMO / "test_wording.py").unlink()

### Зависимость от порядка

Тесты обязаны проходить по отдельности и в любом порядке. Общее изменяемое
состояние это ломает:

In [ ]:
(DEMO / "test_order.py").write_text(
    """collected: list[int] = []


def test_adds() -> None:
    collected.append(1)
    assert collected == [1]


def test_reads() -> None:
    assert collected == [1]
""",
    encoding="utf-8",
)

run_pytest("test_order.py")

Вместе — зелено. А теперь запустим только второй тест, как это сделал бы любой,
кто чинит именно его:

In [ ]:
run_pytest("test_order.py::test_reads")

Тест падает, хотя код не менялся. Он проверял не поведение, а след, оставленный
соседом. Лечится тем же `tmp_path` и фикстурами: состояние должно создаваться
для теста, а не жить на уровне модуля.

## Итого

| Задача | Решение | Чего не делать |
|---|---|---|
| проверить результат | `assert результат == ожидаемое` | вызвать функцию и не проверить ответ |
| много похожих случаев | `@pytest.mark.parametrize` | цикл внутри одного теста |
| ожидаемое падение | `pytest.raises(КонкретныйКласс)` | `pytest.raises(Exception)` |
| дробные числа | `pytest.approx` | точное равенство |
| файлы в тесте | `tmp_path` | настоящие файлы проекта |
| общая обстановка | фикстура, `conftest.py` | копировать подготовку в каждый тест |
| внешние зависимости | `monkeypatch` | менять окружение насовсем |
| имя теста | что проверяется | `test_1`, `test_2` |

Проверка качества самого теста одна: **сломайте код, который он проверяет**.
Тест, остающийся зелёным на сломанном коде, не тест, а строчка в отчёте.

In [ ]:
import shutil

shutil.rmtree(DEMO)
print("прибрались:", not DEMO.exists())